# Universal Adversarial Perturbations - Full Experiment

This notebook runs the complete UAP experiment cycle:
1. Setup and load pre-trained models
2. Compute Universal Adversarial Perturbation (UAP)
3. Run comprehensive analysis (Question 1 & 2)
4. Generate all visualizations

**Estimated runtime on Colab GPU:** 1-2 hours

**Dataset:** CIFAR-100  
**Models:** ResNet-32, VGG-16  
**Subset size:** 1000 images

## 1. Setup and Installation

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Clone repository (if not already cloned)
import os
if not os.path.exists('rsai-a1'):
    !git clone https://git.soma.salesforce.com/mkanaka/rsai-a1.git
    %cd rsai-a1
else:
    %cd rsai-a1
    !git pull

In [ ]:
# Install dependencies
!pip install -q torch torchvision tqdm scikit-learn matplotlib seaborn

In [ ]:
# Import required modules
import sys
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json
from PIL import Image

# Add src to path
sys.path.insert(0, '/content/rsai-a1')

from config import DEVICE, MODEL_NAME, MODEL_PATH, MODEL_NAME_TRANSFER, MODEL_PATH_TRANSFER, PATHS
from src.models import load_cifar100_model
from src.data_utils import get_cifar100_loaders, get_cifar100_subset
from src.uap import compute_uap
from src.analysis import (
    compute_gradient_correlation,
    analyze_decision_boundary_dimensionality,
    test_transferability,
    analyze_perturbation_properties
)
from src.evaluation import (
    visualize_perturbation,
    visualize_correlation_matrix,
    visualize_pca_variance,
    visualize_adversarial_examples,
    save_results_json
)

print("✓ All imports successful")

In [ ]:
# Set device (Colab will use CUDA)
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
    
print(f"Using device: {device}")

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 2. Load Pre-trained Models

In [ ]:
# Load source model (ResNet-32)
print("Loading source model...")
print("This will download pre-trained weights from torch.hub")

# Create checkpoints directory
Path('./checkpoints').mkdir(exist_ok=True)

# Load and save ResNet-32
if not Path('./checkpoints/cifar100_resnet32.pth').exists():
    print("Downloading ResNet-32...")
    model_resnet = torch.hub.load(
        "chenyaofo/pytorch-cifar-models",
        "cifar100_resnet32",
        pretrained=True,
        trust_repo=True
    )
    torch.save(model_resnet.state_dict(), './checkpoints/cifar100_resnet32.pth')
    print("✓ ResNet-32 downloaded and saved")
else:
    print("✓ ResNet-32 checkpoint already exists")

# Load model
source_model = load_cifar100_model('cifar100_resnet32', './checkpoints/cifar100_resnet32.pth', device)
print(f"✓ Source model loaded: ResNet-32")

In [ ]:
# Load target model (VGG-16) for transferability testing
print("Loading target model...")

if not Path('./checkpoints/cifar100_vgg16_bn.pth').exists():
    print("Downloading VGG-16...")
    model_vgg = torch.hub.load(
        "chenyaofo/pytorch-cifar-models",
        "cifar100_vgg16_bn",
        pretrained=True,
        trust_repo=True
    )
    torch.save(model_vgg.state_dict(), './checkpoints/cifar100_vgg16_bn.pth')
    print("✓ VGG-16 downloaded and saved")
else:
    print("✓ VGG-16 checkpoint already exists")

target_model = load_cifar100_model('cifar100_vgg16_bn', './checkpoints/cifar100_vgg16_bn.pth', device)
print(f"✓ Target model loaded: VGG-16")

In [ ]:
# Verify model accuracy on test set
print("Verifying model accuracy...\n")

from src.models import test_model_accuracy

# Load test data
_, test_loader = get_cifar100_loaders(
    data_root='./data',
    batch_size=128,
    num_workers=2
)

# Test ResNet-32
print("Testing ResNet-32...")
resnet_acc = test_model_accuracy(source_model, test_loader, device)
print(f"ResNet-32 Accuracy: {resnet_acc:.2f}%\n")

# Test VGG-16
print("Testing VGG-16...")
vgg_acc = test_model_accuracy(target_model, test_loader, device)
print(f"VGG-16 Accuracy: {vgg_acc:.2f}%\n")

if resnet_acc >= 70 and vgg_acc >= 70:
    print("✓ Both models meet accuracy requirement (≥70%)")
else:
    print("⚠ Warning: Model accuracy below 70%")

## 3. Compute Universal Adversarial Perturbation

This will take **1-2 hours on GPU** (3-4 hours on CPU).  
Target: ≥70% fooling rate on 1000 training images.

In [ ]:
# Load training subset for UAP computation
print("Loading training subset (1000 images)...")

subset_dataset = get_cifar100_subset(
    data_root='./data',
    subset_size=1000,
    train=True,
    seed=42
)

subset_loader = torch.utils.data.DataLoader(
    subset_dataset,
    batch_size=1,  # UAP processes one image at a time
    shuffle=False,
    num_workers=2
)

print(f"✓ Training subset ready: {len(subset_dataset)} images")

In [ ]:
# Compute UAP
print("="*80)
print("COMPUTING UNIVERSAL ADVERSARIAL PERTURBATION")
print("="*80)
print("This will take 1-2 hours on GPU...\n")

# Create results directory
Path('./results').mkdir(exist_ok=True)

# Run UAP computation
uap, fooling_rates = compute_uap(
    model=source_model,
    dataset=subset_dataset,  # Fixed: correct parameter name
    device=device,
    xi=10/255,              # Perturbation budget (L∞)
    delta=0.8,              # Target fooling rate (80%)
    max_iter_uni=10,        # Maximum UAP iterations
    norm_type='inf',        # L∞ norm
    save_interval=1,        # Save every iteration
    save_path='./results'   # Fixed: correct parameter name
)

# Save final UAP
torch.save({
    'perturbation': uap,
    'fooling_rates': fooling_rates,
    'final_fooling_rate': fooling_rates[-1] if fooling_rates else 0,
}, './results/uap_final.pt')

print("\n" + "="*80)
print("UAP COMPUTATION COMPLETE")
print("="*80)
print(f"Final fooling rate: {fooling_rates[-1]:.2f}%")
print(f"UAP saved to: ./results/uap_final.pt")

In [ ]:
# Analyze UAP properties
print("\nAnalyzing UAP properties...\n")
uap_stats = analyze_perturbation_properties(uap, name="Universal Adversarial Perturbation")

# Visualize UAP
visualize_perturbation(
    perturbation=uap,
    save_path='./results/uap_perturbation.png',
    title='Universal Adversarial Perturbation'
)

# Display in notebook
from IPython.display import Image as IPImage, display
display(IPImage('./results/uap_perturbation.png'))

In [ ]:
# Plot fooling rate convergence
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(fooling_rates) + 1), fooling_rates, 'b-o', linewidth=2)
plt.axhline(y=80, color='r', linestyle='--', label='Target (80%)')
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Fooling Rate (%)', fontsize=12)
plt.title('UAP Convergence', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('./results/uap_convergence.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Convergence plot saved to ./results/uap_convergence.png")

## 4. Question 1: Why Does a Single Perturbation Fool Many Images?

We analyze:
1. **Gradient Correlation**: Do different images share similar vulnerability directions?
2. **PCA Dimensionality**: Do decision boundaries lie in a low-dimensional subspace?

In [ ]:
# Gradient Correlation Analysis
print("="*80)
print("GRADIENT CORRELATION ANALYSIS")
print("="*80)
print("Analyzing shared vulnerability directions...\n")

# Get sample images for analysis
analysis_subset = get_cifar100_subset(
    data_root='./data',
    subset_size=100,
    train=True,
    seed=42
)
analysis_loader = torch.utils.data.DataLoader(
    analysis_subset, batch_size=32, shuffle=False
)

images_list = []
labels_list = []
for images, labels in analysis_loader:
    images_list.append(images)
    labels_list.append(labels)
    
images = torch.cat(images_list)
labels = torch.cat(labels_list)

# Compute correlation
mean_correlation, correlation_matrix = compute_gradient_correlation(
    model=source_model,
    images=images,
    labels=labels,
    device=device
)

print(f"\n✓ Mean gradient correlation: {mean_correlation:.4f}")

# Visualize
visualize_correlation_matrix(
    correlation_matrix=correlation_matrix,
    save_path='./results/gradient_correlation.png',
    title=f'Gradient Correlation Matrix (Mean: {mean_correlation:.4f})'
)

display(IPImage('./results/gradient_correlation.png'))

In [ ]:
# PCA Dimensionality Analysis
print("\n" + "="*80)
print("PCA DIMENSIONALITY ANALYSIS")
print("="*80)
print("Analyzing decision boundary dimensionality...\n")

# Use more samples for PCA
pca_subset = get_cifar100_subset(
    data_root='./data',
    subset_size=500,
    train=True,
    seed=42
)
pca_loader = torch.utils.data.DataLoader(
    pca_subset, batch_size=32, shuffle=False
)

images_pca_list = []
labels_pca_list = []
for images, labels in pca_loader:
    images_pca_list.append(images)
    labels_pca_list.append(labels)
    
images_pca = torch.cat(images_pca_list)
labels_pca = torch.cat(labels_pca_list)

# Run PCA
pca_results = analyze_decision_boundary_dimensionality(
    model=source_model,
    images=images_pca,
    labels=labels_pca,
    device=device,
    n_components=50
)

print(f"\n✓ Effective dimension (90% variance): {pca_results['effective_dimension']}/50")
print(f"✓ Total variance captured: {pca_results['total_variance_90pct']:.4f}")

# Visualize
visualize_pca_variance(
    explained_variance=pca_results['explained_variance_ratio'],
    cumulative_variance=pca_results['cumulative_variance'],
    effective_dim=pca_results['effective_dimension'],
    save_path='./results/pca_variance.png'
)

display(IPImage('./results/pca_variance.png'))

In [ ]:
# Question 1 Summary
print("\n" + "="*80)
print("QUESTION 1 SUMMARY")
print("="*80)
print(f"\nGradient Correlation: {mean_correlation:.4f}")
if mean_correlation > 0.5:
    print("  → HIGH correlation: Images share strong vulnerability directions")
elif mean_correlation > 0.2:
    print("  → MODERATE correlation: Some shared vulnerability patterns")
else:
    print("  → LOW correlation: Limited shared vulnerability")

print(f"\nEffective Dimension: {pca_results['effective_dimension']}/50 components")
if pca_results['effective_dimension'] < 10:
    print("  → VERY LOW dimensionality: Decision boundaries highly concentrated")
elif pca_results['effective_dimension'] < 20:
    print("  → LOW dimensionality: Decision boundaries in compact subspace")
else:
    print("  → MODERATE dimensionality: Decision boundaries more spread out")

print("\n" + "="*80)
print("CONCLUSION")
print("="*80)
print("A single UAP can fool multiple images because:")
print(f"  1. Images share vulnerability directions (correlation: {mean_correlation:.4f})")
print(f"  2. Decision boundaries lie in low-dimensional subspace ({pca_results['effective_dimension']} dims)")
print("  3. The UAP exploits this shared geometric structure!")

## 5. Question 2: UAP vs FGSM/PGD Transferability

We test how well UAP transfers from ResNet-32 to VGG-16 compared to per-image attacks.

In [ ]:
# Transferability Testing
print("="*80)
print("TRANSFERABILITY ANALYSIS")
print("="*80)
print(f"Source Model: ResNet-32")
print(f"Target Model: VGG-16")
print(f"Testing on 1000 test images...\n")

# Test UAP transferability
transfer_results = test_transferability(
    perturbation=uap,
    source_model=source_model,
    target_model=target_model,
    test_loader=test_loader,
    device=device,
    max_samples=1000
)

print("\n" + "="*80)
print("TRANSFERABILITY RESULTS")
print("="*80)
print(f"Source fooling rate (ResNet-32): {transfer_results['source_fooling_rate']:.2f}%")
print(f"Target fooling rate (VGG-16):    {transfer_results['target_fooling_rate']:.2f}%")
print(f"Transferability ratio:            {transfer_results['transferability_ratio']:.2f}")

if transfer_results['transferability_ratio'] > 0.7:
    print("\n  ✓ HIGH transferability: UAP transfers well across architectures")
    print("    → Suggests exploitation of fundamental dataset properties")
elif transfer_results['transferability_ratio'] > 0.4:
    print("\n  ✓ MODERATE transferability: UAP partially transfers")
    print("    → Some architecture-specific vulnerabilities")
else:
    print("\n  ✓ LOW transferability: UAP is model-specific")
    print("    → Exploits architecture-specific features")

In [ ]:
# Visualize adversarial examples
print("\nGenerating adversarial examples visualization...")

# Get a batch of test images
test_images, test_labels = next(iter(test_loader))
test_images = test_images[:5].to(device)
test_labels = test_labels[:5].to(device)

# Get predictions
with torch.no_grad():
    # Clean predictions
    outputs_clean = source_model(test_images)
    _, preds_clean = outputs_clean.max(1)
    
    # Perturbed predictions
    uap_batch = uap.unsqueeze(0).expand(5, -1, -1, -1)
    perturbed_images = torch.clamp(test_images + uap_batch, 0, 1)
    outputs_pert = source_model(perturbed_images)
    _, preds_pert = outputs_pert.max(1)

# Visualize
visualize_adversarial_examples(
    images=test_images,
    perturbation=uap,
    original_preds=preds_clean,
    perturbed_preds=preds_pert,
    labels=test_labels,
    n_examples=5,
    save_path='./results/adversarial_examples.png'
)

display(IPImage('./results/adversarial_examples.png'))

## 6. Save All Results

In [ ]:
# Compile all results
all_results = {
    'uap_computation': {
        'final_fooling_rate': float(fooling_rates[-1]),
        'convergence': [float(fr) for fr in fooling_rates],
        'l2_norm': float(uap_stats['l2_norm']),
        'linf_norm': float(uap_stats['linf_norm']),
    },
    'question1': {
        'gradient_correlation': {
            'mean_correlation': float(mean_correlation),
        },
        'pca_dimensionality': {
            'effective_dimension': int(pca_results['effective_dimension']),
            'total_variance_90pct': float(pca_results['total_variance_90pct']),
        }
    },
    'question2': {
        'transferability': transfer_results
    },
    'models': {
        'source': 'ResNet-32',
        'target': 'VGG-16',
        'source_accuracy': float(resnet_acc),
        'target_accuracy': float(vgg_acc),
    },
    'config': {
        'dataset': 'CIFAR-100',
        'subset_size': 1000,
        'xi': 10/255,
        'norm_type': 'inf',
        'device': str(device),
    }
}

# Save as JSON
save_results_json(all_results, './results/experiment_results.json')

print("\n" + "="*80)
print("ALL RESULTS SAVED")
print("="*80)
print("\nGenerated files:")
print("  - uap_final.pt                   (Trained UAP)")
print("  - uap_perturbation.png           (UAP visualization)")
print("  - uap_convergence.png            (Training convergence)")
print("  - gradient_correlation.png       (Question 1)")
print("  - pca_variance.png               (Question 1)")
print("  - adversarial_examples.png       (Example fooling)")
print("  - experiment_results.json        (All metrics)")
print("\n" + "="*80)

## 7. Final Summary

In [ ]:
# Print comprehensive summary
print("="*80)
print("EXPERIMENT COMPLETE - FINAL SUMMARY")
print("="*80)

print("\n📊 UAP PERFORMANCE:")
print(f"  • Final fooling rate: {fooling_rates[-1]:.2f}%")
print(f"  • L∞ norm: {uap_stats['linf_norm']:.6f} (budget: {10/255:.6f})")
print(f"  • L2 norm: {uap_stats['l2_norm']:.4f}")

print("\n🔍 QUESTION 1: Why does a single UAP fool many images?")
print(f"  • Gradient correlation: {mean_correlation:.4f}")
print(f"  • Effective dimension: {pca_results['effective_dimension']}/50 components")
print(f"  • Conclusion: Shared vulnerabilities in low-dimensional subspace")

print("\n🔄 QUESTION 2: UAP Transferability")
print(f"  • Source (ResNet-32): {transfer_results['source_fooling_rate']:.2f}%")
print(f"  • Target (VGG-16): {transfer_results['target_fooling_rate']:.2f}%")
print(f"  • Transfer ratio: {transfer_results['transferability_ratio']:.2f}")

print("\n✅ SUCCESS CRITERIA:")
criteria_met = []
if fooling_rates[-1] >= 70:
    criteria_met.append("  ✓ Fooling rate ≥70%")
else:
    criteria_met.append("  ✗ Fooling rate <70%")
    
if uap_stats['linf_norm'] <= 10/255:
    criteria_met.append("  ✓ L∞ norm within budget")
else:
    criteria_met.append("  ✗ L∞ norm exceeds budget")
    
criteria_met.append("  ✓ Analysis complete")
criteria_met.append("  ✓ Visualizations generated")

for criterion in criteria_met:
    print(criterion)

print("\n" + "="*80)
print("All results saved to ./results/")
print("Download the results folder to include in your report!")
print("="*80)

In [ ]:
# Create zip file for easy download
import shutil

print("Creating results archive...")
shutil.make_archive('uap_experiment_results', 'zip', './results')
print("\n✓ Results archived to: uap_experiment_results.zip")
print("\nDownload this file from the Colab file browser!")

# Display file list
print("\nContents:")
!ls -lh results/